## Langchain Expression Language
* Simple syntax to inference LLM, Prompt, Output Parser
* Langchain streaming : Returns word by word
* Batching : Give two or more queries at a time
* Async Operation 

In [3]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in one sentence."
)

parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"topic": "Investment Banking"})

print(result)

Investment banking helps corporations, governments, and institutions raise capital and execute complex financial transactions—such as mergers, acquisitions, initial public offerings, and debt/equity issuances—by acting as an intermediary between security issuers and investors.


## RunnableLambda


In [12]:
from langchain_core.runnables import RunnableLambda

def add_exclamation(text):
    return text + "!!!"

chain2 = prompt | llm | parser | RunnableLambda(add_exclamation)

print(chain2.invoke({"topic": "Deep Learning"}))

Deep learning is a subset of machine learning that uses multi‑layered neural networks to automatically learn hierarchical representations from raw data.!!!


## RunnablePassthrough
* Pass the input to multiple steps

In [15]:
from langchain_core.runnables import RunnablePassthrough

chain3 = (
    {"topic": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

chain3.invoke("AI Agents")

'An AI agent is a system that perceives its environment through sensors and acts upon it via effectors to autonomously achieve specific goals. (24 words)'

## RunnableParallel
* Run the multiple LLM task at once

In [20]:
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import PromptTemplate

joke_chain = PromptTemplate.from_template(
    "Tell a joke about {topic}"
) | llm | parser

fact_chain = PromptTemplate.from_template(
    "Give a fact about {topic}"
) | llm | parser

chain4 = RunnableParallel(
    joke=joke_chain,
    fact=fact_chain
)

print(chain4.invoke({"topic": "AI"}))

{'joke': 'Why did the AI cross the road?  \nTo get to the other side… but first it recalculated the route 47 times because it detected a *slightly* uneven pavement crack and wanted to optimize for "maximum pedestrian joy." 😄  \n\n*(Bonus groaner: It still ended up stuck in a loop asking, "Did you mean *the* other side, or *a* other side?")*  \n\nHope that sparked a smile! AI humor is all about finding the funny in how literally we take our algorithms… until they start judging our life choices. 😉', 'fact': "Here's a concrete, well-documented fact about AI that highlights a significant real-world impact:\n\n**Training a single large AI model like GPT-3 can consume as much electricity as 120 average U.S. homes use in an entire year.**\n\n### Details:\n- According to a 2021 study by Stanford University's Human-Centered AI Institute (HAI), training GPT-3 (with 175 billion parameters) required approximately **1,287 megawatt-hours (MWh)** of electricity.\n- For context: The average U.S. house

## Streaming :
* Streaming is a generator and it returns the object

In [27]:
for chunk in chain.stream({"topic": "Robotics"}):
    print(chunk, end="", flush=True)

Robotics is the interdisciplinary field focused on designing, building, and programming machines (robots) that can sense, process information, and act autonomously or semi-autonomously in the physical world to perform tasks.

## Batching :
* Give two or more queries at a time 

In [34]:

prompt4 = PromptTemplate.from_template(
    "Answer the question: {question}"
)

parser4 = StrOutputParser()

chain5 = prompt4 | llm | parser4


batch_response = chain5.batch([
    {"question": "what are neural networks"},
    {"question": "what is machine learning"}
])

print(batch_response)

['Neural networks are a type of **machine learning model inspired by the structure and function of the human brain**, though they are *mathematical abstractions* and **not** accurate simulations of biological brains. Here\'s a clear, concise breakdown:\n\n### 🔑 Core Idea:\n- They consist of interconnected nodes (called **"neurons"** or **"units"**) organized in **layers**.\n- Data flows through these layers: an **input layer** → one or more **hidden layers** → an **output layer**.\n- Each connection between neurons has a **weight** (a number that adjusts during learning), and neurons apply a simple mathematical function to the weighted sum of their inputs.\n\n### 🧠 How They Learn (Simplified):\n1. **Input**: Raw data (e.g., pixels of an image, words in a sentence) enters the input layer.\n2. **Processing**: In each hidden layer, neurons combine inputs from the previous layer using their weights, apply an activation function (e.g., ReLU, sigmoid), and pass results forward.\n3. **Output*